## Загрузка библотек и подключение к Spark

In [20]:
# -*- coding: utf-8 -*-

# Установка Spark и Java
!apt-get update
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.3.0/spark-3.3.0-bin-hadoop3.tgz
!tar xf spark-3.3.0-bin-hadoop3.tgz
!pip install -q findspark pyspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.3.0-bin-hadoop3"

import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from math import radians, sin, cos, sqrt, asin

# Создание сессии Spark
spark = SparkSession.builder \
    .appName("SF_Bike_Share_Analysis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

trips_df = spark.read.csv("/content/trip.csv", header=True, inferSchema=True)
stations_df = spark.read.csv("/content/station.csv", header=True, inferSchema=True)


Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Ign:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Ign:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Ign:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Ign:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Ign:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Ign:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Err:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
  503  Service Unavailable 

## 1. Найти велосипед с максимальным временем пробега

In [21]:
bike_stats = trips_df.groupBy("bike_id") \
                     .agg(sum("duration").alias("total_duration_sec")) \
                     .orderBy(col("total_duration_sec").desc())

max_bike = bike_stats.first()

print(f"1. Велосипед с ID {max_bike['bike_id']} имеет максимальное суммарное время пробега: {max_bike['total_duration_sec']:,} секунд")
print(f"Это составляет {max_bike['total_duration_sec'] / 3600:.2f} часов ({max_bike['total_duration_sec'] / 86400:.2f} дней)")


1. Велосипед с ID 535 имеет максимальное суммарное время пробега: 18,611,693 секунд
Это составляет 5169.91 часов (215.41 дней)


## 2. Найти наибольшее геодезическое расстояние между станциями

In [22]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Радиус Земли в км

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))

    return R * c

# Собираем станции с координатами на драйвер
stations_local = stations_df.select("id", "name", "lat", "long") \
                            .filter(col("lat").isNotNull() & col("long").isNotNull()) \
                            .collect()

# Вычисляем все попарные расстояния локально
max_distance = 0
max_pair = None

for i in range(len(stations_local)):
    for j in range(i+1, len(stations_local)):
        s1 = stations_local[i]
        s2 = stations_local[j]

        dist = haversine_distance(s1['lat'], s1['long'], s2['lat'], s2['long'])

        if dist > max_distance:
            max_distance = dist
            max_pair = (s1, s2)

print(f"Станция 1: {max_pair[0]['name']} (ID: {max_pair[0]['id']})")
print(f"  Координаты: {max_pair[0]['lat']}, {max_pair[0]['long']}")
print(f"Станция 2: {max_pair[1]['name']} (ID: {max_pair[1]['id']})")
print(f"  Координаты: {max_pair[1]['lat']}, {max_pair[1]['long']}")
print(f"Расстояние: {max_distance:.2f} км")


Станция 1: SJSU - San Salvador at 9th (ID: 16)
  Координаты: 37.333954999999996, -121.877349
Станция 2: Embarcadero at Sansome (ID: 60)
  Координаты: 37.80477, -122.40323400000001
Расстояние: 69.92 км


## 3. Найти путь велосипеда с максимальным временем пробега через станции

In [23]:
winner_bike_id = max_bike['bike_id']
print(f"Анализируем путь велосипеда #{winner_bike_id}")

# Фильтруем поездки велосипеда-победителя и сортируем по времени
bike_trips = trips_df.filter(col("bike_id") == winner_bike_id) \
                     .orderBy("start_date")

trip_count = bike_trips.count()
print(f"Количество поездок велосипеда #{winner_bike_id}: {trip_count}")

if trip_count > 0:
    stations_chain = []

    for i, trip in enumerate(trips_local):
        if i == 0:
            stations_chain.append(trip['start_station_name'])
        stations_chain.append(trip['end_station_name'])

    print(f"\nПОЛНЫЙ МАРШРУТ ВЕЛОСИПЕДА #{winner_bike_id}")
    print(" → ".join(stations_chain))
else:
    print(f"Велосипед {winner_bike_id} не совершал поездок")


Анализируем путь велосипеда #535
Количество поездок велосипеда #535: 1328

ПОЛНЫЙ МАРШРУТ ВЕЛОСИПЕДА #535
Mechanics Plaza (Market at Battery) → Embarcadero at Sansome → Market at 4th → South Van Ness at Market → Powell Street BART → San Francisco Caltrain (Townsend at 4th) → Temporary Transbay Terminal (Howard at Beale) → Market at 10th → Market at 10th → Market at 4th → San Francisco Caltrain (Townsend at 4th) → Beale at Market → Davis at Jackson → Embarcadero at Vallejo → Market at Sansome → Davis at Jackson → 2nd at South Park → 2nd at Folsom → 2nd at South Park → San Francisco Caltrain (Townsend at 4th) → Embarcadero at Bryant → San Francisco Caltrain (Townsend at 4th) → South Van Ness at Market → Howard at 2nd → Market at Sansome → 2nd at South Park → Howard at 2nd → Market at 4th → Golden Gate at Polk → San Francisco Caltrain (Townsend at 4th) → 2nd at Folsom → San Francisco Caltrain 2 (330 Townsend) → San Francisco Caltrain (Townsend at 4th) → Embarcadero at Vallejo → Market at 

## 4. Найти количество велосипедов в системе


In [24]:
unique_bikes = trips_df.select("bike_id").distinct().count()
print(f" Уникальных велосипедов в системе: {unique_bikes:,}")

 Уникальных велосипедов в системе: 700


## 5. Найти пользователей потративших на поездки более 3 часов

In [25]:
# 3 часа = 10800 секунд
THRESHOLD_SECONDS = 10800

# Анализ по ZIP коду
user_by_zip = trips_df.groupBy("zip_code") \
                      .agg(
                          sum("duration").alias("total_duration_sec"),
                          count("*").alias("num_trips"),
                          avg("duration").alias("avg_duration_sec")
                      ) \
                      .filter(col("total_duration_sec") > THRESHOLD_SECONDS) \
                      .orderBy(col("total_duration_sec").desc())

user_count = user_by_zip.count()
print(f"5. Пользователей с общим временем >3 часов: {user_count}")

# Остановка Spark сессии
spark.stop()


5. Пользователей с общим временем >3 часов: 3661
